## Text Generation using Gated Recurrent Unit Networks - ML


Gated Recurrent Unit (GRU) are a type of Recurrent Neural Network (RNN) that are designed to handle sequential data such as text by using gating mechanisms to regulate the flow of information. Unlike traditional RNNs which suffer from vanishing gradient problems, GRUs offer a more efficient way to capture long-range dependencies in sequences. In this article, we will learn to build a Text Generator using a GRU network to generate creative text based on the learned patterns.

### 1. Importing the required libraries

We will import the following libraries :

* **pandas:** For easy data loading and handling.
* **numpy:** For numerical operations.
* **tensorflow:** To build and train the GRU model.
* **random:** For generating random starting points in text.



In [2]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, Activation
from tensorflow.keras.optimizers import RMSprop
import random

### 2. Loading the data into a string

Here we are using a dataset of poems to train our GRU model. You can download dataset from here. We load the text lines into a pandas Data Frame, join all lines into one string and preview the first 500 characters.

In [4]:
dataset = "https://raw.githubusercontent.com/itsluckysharma01/Datasets/refs/heads/main/poems.csv"

df = pd.read_csv(dataset, header=None, names=['text'])
text = " ".join(df['text'].tolist())

print(text[:500])

Through the forest deep, where shadows linger long, the night sings its song. In the stillness of night, the moon whispers its light. Under the velvet sky, dreams take flight on wings of stars. A single leaf falls, carried away by the gentle breeze. In the stillness of night, the moon whispers its light. Love is the light that guides us through the darkest times. A single leaf falls, carried away by the gentle breeze. A whisper of hope in the silence of the dawn. A whisper of hope in the silence


### 3. Creating Character Mappings

We will extract unique characters in the text and create mappings from characters to indices and back.

In [5]:
vocab = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(vocab)}
idx_to_char = {i: c for i, c in enumerate(vocab)}

### 4. Prepare Input and Output Sequences

We will split the text into overlapping sequences of length 100. For each sequence, the next character is the label. Then we also One-hot encode inputs and outputs.

In [6]:
max_len = 100
step = 5

sentences = []
next_chars = []

for i in range(0, len(text) - max_len, step):
    sentences.append(text[i: i + max_len])
    next_chars.append(text[i + max_len])

X = np.zeros((len(sentences), max_len, len(vocab)), dtype=bool)
y = np.zeros((len(sentences), len(vocab)), dtype=bool)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_idx[char]] = 1
    y[i, char_to_idx[next_chars[i]]] = 1

### 5. Building the GRU network

In [7]:
model = Sequential()
model.add(GRU(128, input_shape=(max_len, len(vocab))))
model.add(Dense(len(vocab)))
model.add(Activation('softmax'))

optimizer = RMSprop(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

model.summary()

c:\Users\PANDIT JI\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 128)            │        63,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 35)             │         4,515 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 35)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,875 (265.14 KB)

 Trainable params: 67,875 (265.14 KB)

 Non-trainable params: 0 (0.00 B)

### 6. Training the GRU model

The model.fit() function trains the model on the input data (X) and target labels (y) for 30 epochs with a batch size of 128.

In [ ]:
model.fit(X, y, batch_size=128, epochs=30)

### 7. Defining Text Generation Function

In [ ]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-8) / temperature  # add epsilon to avoid log(0)
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(length=400, temperature=0.5):
    start_idx = random.randint(0, len(text) - max_len - 1)
    generated = ''
    sentence = text[start_idx: start_idx + max_len]
    generated += sentence

    for _ in range(length):
        x_pred = np.zeros((1, max_len, len(vocab)))
        for t, char in enumerate(sentence):
            x_pred[0, t, char_to_idx[char]] = 1
        
        preds = model.predict(x_pred, verbose=0)[0]
        next_idx = sample(preds, temperature)
        next_char = idx_to_char[next_idx]

        generated += next_char
        sentence = sentence[1:] + next_char

    return generated

### 8. Generate Sample Text
We can generate same text using our trained model now.

In [ ]:
print(generate_text(length=500, temperature=0.3))